# 6B Feature 가용성 검토

## tl;dr

완료·공식 연승 결과 경주는 4,582개다. 사후 `is_valid_start` 프록시로 48,524개 학습 후보 행과 13,740개 양성을 구성할 수 있고 공식 적중마 미결합은 0건이다. 다만 DNS 659행이 있는 605경주(13.2%)는 발매 마감 당시 모집단을 확정할 수 없으므로, 전체 프록시와 DNS 없는 3,977경주·42,632행을 병행 비교해야 한다. 이 Notebook은 DuckDB를 읽기 전용으로 조회하며 Feature 테이블이나 모델을 생성하지 않는다.

## Context & Methods

### Key Assumptions

- 6A 계약에 따라 예측시점은 연승 발매 마감 직전이다.
- 공식 PLC 적중 원문이 있는 완료 경주만 타깃 검사 대상이다.
- `is_valid_start`는 사후 출전 결과이지 발매 마감 당시 베팅 가능 여부의 직접 증거가 아니다.
- 현재 경주 결과 필드는 Feature 후보에서 제외한다.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'PROJECT_CHARTER.md').exists():
    project_root = project_root.parent
database_path = project_root / 'data' / 'warehouse' / 'kra.duckdb'
connection = duckdb.connect(str(database_path), read_only=True)
print(f'Source: {database_path}')

Source: C:\Users\jjk00\Documents\GitHub\kra-racing-analytics\data\warehouse\kra.duckdb


## Data

### 1. 모집단 복원 가능 범위

In [2]:
population_summary = connection.sql("""
WITH plc_races AS (
    SELECT DISTINCT race_id
    FROM canonical.winning_payout
    WHERE pool_code = '연식' AND parse_status = 'PARSED'
), race_profile AS (
    SELECT r.race_id, r.race_date, r.meet_name, r.race_grade, r.runner_count,
           count(*) AS result_rows,
           count(*) FILTER (WHERE rr.result_status = 'FINISHED') AS finished_rows,
           count(*) FILTER (WHERE rr.is_valid_start) AS valid_start_rows,
           count(*) FILTER (WHERE rr.result_status != 'FINISHED') AS non_finished_rows,
           count(*) FILTER (WHERE rr.result_status IN ('DNS', 'SCRATCHED', 'EXCLUDED',
               'NON_STANDARD_UNRESOLVED', 'CANCELLED_STOPPED_OR_DISQUALIFIED')) AS eligibility_ambiguous_rows
    FROM canonical.race r
    JOIN plc_races p USING (race_id)
    JOIN canonical.runner_result rr USING (race_id)
    WHERE r.race_status = 'COMPLETED'
    GROUP BY ALL
)
SELECT count(*) AS plc_completed_races,
       sum(result_rows) AS runner_rows,
       sum(valid_start_rows) AS valid_start_rows,
       sum(non_finished_rows) AS non_finished_rows,
       count(*) FILTER (WHERE non_finished_rows = 0) AS all_finished_races,
       count(*) FILTER (WHERE eligibility_ambiguous_rows > 0) AS eligibility_ambiguous_races,
       sum(eligibility_ambiguous_rows) AS eligibility_ambiguous_rows,
       count(*) FILTER (WHERE eligibility_ambiguous_rows = 0) AS clean_eligibility_races,
       sum(valid_start_rows) FILTER (WHERE eligibility_ambiguous_rows = 0) AS clean_valid_start_rows
FROM race_profile
""").df()
display(population_summary)

status_profile = connection.sql("""
WITH plc_races AS (
    SELECT DISTINCT race_id FROM canonical.winning_payout
    WHERE pool_code = '연식' AND parse_status = 'PARSED'
)
SELECT rr.result_status, rr.is_valid_start, rr.is_valid_finish,
       count(*) AS runner_rows, count(DISTINCT rr.race_id) AS races
FROM canonical.runner_result rr
JOIN canonical.race r USING (race_id)
JOIN plc_races p USING (race_id)
WHERE r.race_status = 'COMPLETED'
GROUP BY ALL ORDER BY runner_rows DESC
""").df()
display(status_profile)

,plc_completed_races,runner_rows,valid_start_rows,non_finished_rows,all_finished_races,eligibility_ambiguous_races,eligibility_ambiguous_rows,clean_eligibility_races,clean_valid_start_rows
0,4582,49183.0,48524.0,795.0,3871,605,659.0,3977,42632.0


,result_status,is_valid_start,is_valid_finish,runner_rows,races
0,FINISHED,True,True,48388,4582
1,DNS,False,False,659,605
2,RACE_STOPPED,True,False,133,119
3,DISQUALIFIED,True,False,3,3


### 2. 공식 PLC 타깃 결합 무결성

In [3]:
target_integrity = connection.sql("""
WITH plc AS (
    SELECT race_id, horse_no_1 AS gate_no
    FROM canonical.winning_payout
    WHERE pool_code = '연식' AND parse_status = 'PARSED'
), joined AS (
    SELECT p.race_id, p.gate_no, rr.horse_id, rr.result_status, rr.is_valid_start
    FROM plc p
    LEFT JOIN canonical.runner_result rr
      ON p.race_id = rr.race_id AND p.gate_no = rr.gate_no
)
SELECT count(*) AS official_positive_rows,
       count(*) FILTER (WHERE horse_id IS NULL) AS unmatched_positive_rows,
       count(*) FILTER (WHERE horse_id IS NOT NULL AND NOT is_valid_start) AS invalid_start_positives,
       count(DISTINCT race_id) AS races,
       min(gate_no) AS min_gate_no, max(gate_no) AS max_gate_no
FROM joined
""").df()
display(target_integrity)

,official_positive_rows,unmatched_positive_rows,invalid_start_positives,races,min_gate_no,max_gate_no
0,13740,0,0,4582,1,16


## Results

### 3. 현재 컬럼의 완전성과 이력 깊이

In [4]:
column_completeness = connection.sql("""
WITH population AS (
    SELECT r.race_id, r.race_date, r.meet_code, r.race_grade, r.distance_m,
           rr.horse_id, rr.jockey_id, rr.trainer_id, rr.owner_id, rr.gate_no,
           rr.horse_sex, rr.horse_age, rr.carried_weight, rr.horse_weight
    FROM canonical.race r
    JOIN canonical.runner_result rr USING (race_id)
    JOIN (SELECT DISTINCT race_id FROM canonical.winning_payout
          WHERE pool_code = '연식' AND parse_status = 'PARSED') p USING (race_id)
    WHERE r.race_status = 'COMPLETED' AND rr.is_valid_start
)
SELECT count(*) AS rows,
       count(*) FILTER (WHERE race_grade IS NULL OR trim(race_grade) = '') AS race_grade_missing,
       count(*) FILTER (WHERE distance_m IS NULL) AS distance_missing,
       count(*) FILTER (WHERE horse_id IS NULL OR trim(horse_id) = '') AS horse_id_missing,
       count(*) FILTER (WHERE jockey_id IS NULL OR trim(jockey_id) = '') AS jockey_id_missing,
       count(*) FILTER (WHERE trainer_id IS NULL OR trim(trainer_id) = '') AS trainer_id_missing,
       count(*) FILTER (WHERE owner_id IS NULL OR trim(owner_id) = '') AS owner_id_missing,
       count(*) FILTER (WHERE gate_no IS NULL) AS gate_missing,
       count(*) FILTER (WHERE horse_sex IS NULL OR trim(horse_sex) = '') AS sex_missing,
       count(*) FILTER (WHERE horse_age IS NULL) AS age_missing,
       count(*) FILTER (WHERE carried_weight IS NULL) AS carried_weight_missing,
       count(*) FILTER (WHERE horse_weight IS NULL) AS horse_weight_missing
FROM population
""").df()
display(column_completeness.T.rename(columns={0: 'count'}))

history_depth = connection.sql("""
WITH population AS (
    SELECT r.race_id, r.race_date, rr.horse_id, rr.jockey_id, rr.trainer_id
    FROM canonical.race r
    JOIN canonical.runner_result rr USING (race_id)
    JOIN (SELECT DISTINCT race_id FROM canonical.winning_payout
          WHERE pool_code = '연식' AND parse_status = 'PARSED') p USING (race_id)
    WHERE r.race_status = 'COMPLETED' AND rr.is_valid_start
), histories AS (
    SELECT p.*,
      (SELECT count(*) FROM canonical.runner_result hrr JOIN canonical.race hr USING (race_id)
       WHERE hrr.horse_id = p.horse_id AND hr.race_date < p.race_date AND hrr.is_valid_start) AS horse_prior_starts,
      (SELECT count(*) FROM canonical.runner_result hrr JOIN canonical.race hr USING (race_id)
       WHERE hrr.jockey_id = p.jockey_id AND hr.race_date < p.race_date AND hrr.is_valid_start) AS jockey_prior_starts,
      (SELECT count(*) FROM canonical.runner_result hrr JOIN canonical.race hr USING (race_id)
       WHERE hrr.trainer_id = p.trainer_id AND hr.race_date < p.race_date AND hrr.is_valid_start) AS trainer_prior_starts
    FROM population p
)
SELECT count(*) AS rows,
       count(*) FILTER (WHERE horse_prior_starts = 0) AS no_horse_history,
       count(*) FILTER (WHERE jockey_prior_starts = 0) AS no_jockey_history,
       count(*) FILTER (WHERE trainer_prior_starts = 0) AS no_trainer_history,
       median(horse_prior_starts) AS horse_history_median,
       median(jockey_prior_starts) AS jockey_history_median,
       median(trainer_prior_starts) AS trainer_history_median
FROM histories
""").df()
display(history_depth.T.rename(columns={0: 'value'}))

,count
rows,48524
race_grade_missing,0
distance_missing,0
horse_id_missing,0
jockey_id_missing,0
trainer_id_missing,0
owner_id_missing,0
gate_missing,0
sex_missing,0
age_missing,0


,value
rows,48524.0
no_horse_history,4973.0
no_jockey_history,334.0
no_trainer_history,249.0
horse_history_median,6.0
jockey_history_median,328.0
trainer_history_median,320.0


### 4. 월별 학습 후보 규모

In [5]:
monthly_population = connection.sql("""
WITH positives AS (
    SELECT race_id, horse_no_1 AS gate_no
    FROM canonical.winning_payout WHERE pool_code = '연식' AND parse_status = 'PARSED'
), population AS (
    SELECT date_trunc('month', r.race_date)::DATE AS month, r.race_id, rr.horse_id,
           CASE WHEN p.gate_no IS NULL THEN 0 ELSE 1 END AS place_hit
    FROM canonical.race r JOIN canonical.runner_result rr USING (race_id)
    JOIN (SELECT DISTINCT race_id FROM positives) pr USING (race_id)
    LEFT JOIN positives p ON p.race_id = rr.race_id AND p.gate_no = rr.gate_no
    WHERE r.race_status = 'COMPLETED' AND rr.is_valid_start
)
SELECT month, count(DISTINCT race_id) AS races, count(*) AS rows,
       sum(place_hit) AS positives, avg(place_hit) AS positive_rate
FROM population GROUP BY month ORDER BY month
""").df()
display(monthly_population)

,month,races,rows,positives,positive_rate
0,2024-01-01,144,1503,433.0,0.288090
1,2024-02-01,108,1252,324.0,0.258786
2,2024-03-01,180,1952,540.0,0.276639
3,2024-04-01,144,1526,430.0,0.281782
4,2024-05-01,152,1557,455.0,0.292229
5,2024-06-01,172,1691,516.0,0.305145
6,2024-07-01,127,1288,381.0,0.295807
7,2024-08-01,155,1537,464.0,0.301887
8,2024-09-01,123,1328,369.0,0.277861
9,2024-10-01,144,1501,432.0,0.287808


## Takeaways

- 4,582개 완료·공식 연승 경주에서 사후 `is_valid_start` 프록시는 48,524행이며 공식 양성 13,740행은 모두 출전마와 결합된다.
- DNS 659행이 있는 605경주(13.2%)는 발매 마감 당시 베팅 가능 여부를 완전히 입증할 수 없다. DNS 없는 민감도 집합은 3,977경주·42,632행이다.
- 후보 컬럼 중 마체중은 48,524행 전부 결측이다. 나머지 기본 식별·경주·출전 컬럼은 프록시 모집단에서 결측이 없지만 현 경주 값의 사전 가용성은 별도로 입증해야 한다.
- 경주일 미만 과거 이력은 계산할 수 있으나 말 이력이 없는 행이 4,973개(10.2%)이며 수집 시작점에 따른 좌측 절단을 표시해야 한다.
- 추가 정보는 없으면 타깃 모집단이나 운영 예측이 성립하지 않는 `필수`, 첫 기준모델의 설명력을 직접 보강하는 `우선`, 이후 성능 비교로 판단할 `선택`으로 구분한다.

In [6]:
connection.close()